# SPN16: par de textos y reversa de la ultima ronda

Notebook corto para:

- elegir dos textos claros y cifrarlos con `cryptosystems/spn16.py`,
- ver el estado de ambos textos en cada fase y su diferencia XOR,
- elegir subclaves candidatas `K_r` y `K_{r+1}` para revertir la ultima ronda paso a paso.


In [59]:
#from cryptosystems import spn16
import spn16

def fmt16(x: int) -> str:
    return f"0x{x & 0xFFFF:04X}"

def pair_rows(label: str, a: int, b: int):
    return (label, fmt16(a), fmt16(b), fmt16(a ^ b))

def print_pair_trace(title: str, rows):
    print(title)
    print(f"{'Fase':<28} {'Texto A':<10} {'Texto B':<10} {'XOR':<10}")
    print('-' * 64)
    for label, a, b, dx in rows:
        print(f"{label:<28} {a:<10} {b:<10} {dx:<10}")

def print_pair_trace_only_perm(title: str, rows):
    print(title)
    print(f"{'Fase':<28} {'Texto A':<10} {'Texto B':<10} {'XOR':<10}")
    print('-' * 64)
    for label, a, b, dx in rows:
        if "Perm" in label or "Cifrado" in label or "claro" in label:
            print(f"{label:<28} {a:<10} {b:<10} {dx:<10}")
            
def trace_encrypt_block(x: int, subkeys, rounds: int):
    rows = [('Texto claro', x)]
    state = x & 0xFFFF
    for i in range(rounds - 1):
        state ^= subkeys[i]
        rows.append((f'R{i+1} despues XOR K{i+1}', state))
        state = spn16._sub_bytes16(state)
        rows.append((f'R{i+1} despues S-box', state))
        state = spn16._permute16(state)
        rows.append((f'R{i+1} despues Perm', state))

    state ^= subkeys[rounds - 1]
    rows.append((f'R{rounds} despues XOR K{rounds}', state))
    state = spn16._sub_bytes16(state)
    rows.append((f'R{rounds} despues S-box', state))
    state ^= subkeys[rounds]
    rows.append((f'Cifrado despues XOR K{rounds+1}', state))
    return rows

def show_encryption_pair_trace(p1: int, p2: int, subkeys, rounds: int, only_perm: bool = False):
    trace_a = trace_encrypt_block(p1, subkeys, rounds)
    trace_b = trace_encrypt_block(p2, subkeys, rounds)
    rows = [pair_rows(label_a, value_a, value_b) for (label_a, value_a), (_, value_b) in zip(trace_a, trace_b)]
    if only_perm:
        print_pair_trace_only_perm('Cifrado del par con SPN16', rows)
    else:
        print_pair_trace('Cifrado del par con SPN16', rows)
    return trace_a, trace_b

def reverse_last_round_trace(ciphertext: int, candidate_k_r: int, candidate_k_final: int, rounds: int):
    rows = [('Texto cifrado', ciphertext & 0xFFFF)]
    state = (ciphertext ^ candidate_k_final) & 0xFFFF
    rows.append((f'Deshacer XOR K{rounds+1} candidato', state))
    state = spn16._inv_sub_bytes16(state)
    rows.append((f'Aplicar S-box^-1 de R{rounds}', state))
    state ^= candidate_k_r
    rows.append((f'Deshacer XOR K{rounds} candidato', state))
    return rows

def show_reverse_last_round_pair(c1: int, c2: int, candidate_k_r: int, candidate_k_final: int, rounds: int):
    trace_a = reverse_last_round_trace(c1, candidate_k_r, candidate_k_final, rounds)
    trace_b = reverse_last_round_trace(c2, candidate_k_r, candidate_k_final, rounds)
    rows = [pair_rows(label_a, value_a, value_b) for (label_a, value_a), (_, value_b) in zip(trace_a, trace_b)]
    print_pair_trace('Reversa de la ultima ronda con subclaves candidatas', rows)
    return trace_a, trace_b


In [67]:
# Edita estos valores
rounds = 4
master_key_hex = '00112233445566778899'  # para 4 rondas se necesitan 5 subclaves
plain_a = 0x002B
plain_b = 0x0B2B

subkeys = spn16.expand_key_from_hex(master_key_hex, rounds)
cipher_a = spn16.spn_encrypt_block(plain_a, subkeys, rounds)
cipher_b = spn16.spn_encrypt_block(plain_b, subkeys, rounds)

print(f'K1..K{rounds+1} reales: {[fmt16(k) for k in subkeys]}')
print()
trace_a, trace_b = show_encryption_pair_trace(plain_a, plain_b, subkeys, rounds, only_perm=False)
print()
print(f'Cifrado A = {fmt16(cipher_a)}')
print(f'Cifrado B = {fmt16(cipher_b)}')
print(f'Delta C  = {fmt16(cipher_a ^ cipher_b)}')


K1..K5 reales: ['0x0011', '0x2233', '0x4455', '0x6677', '0x8899']

Cifrado del par con SPN16
Fase                         Texto A    Texto B    XOR       
----------------------------------------------------------------
Texto claro                  0x002B     0x0B2B     0x0B00    
R1 despues XOR K1            0x003A     0x0B3A     0x0B00    
R1 despues S-box             0xEE16     0xEC16     0x0200    
R1 despues Perm              0xCDD2     0xCD92     0x0040    
R2 despues XOR K2            0xEFE1     0xEFA1     0x0040    
R2 despues S-box             0x0704     0x0764     0x0060    
R2 despues Perm              0x0544     0x0764     0x0220    
R3 despues XOR K3            0x4111     0x4331     0x0220    
R3 despues S-box             0x2444     0x2114     0x0550    
R3 despues Perm              0x0780     0x0186     0x0606    
R4 despues XOR K4            0x61F7     0x67F1     0x0606    
R4 despues S-box             0xB478     0xB874     0x0C0C    
Cifrado despues XOR K5       0x3CE1 

In [61]:
def print_pair_trace_rounds(title: str, rows):
    print(title)
    print(f"{'Fase':<28} {'Texto A':<10} {'Texto B':<10} {'XOR':<10}")
    print('-' * 64)
    for label, a, b, dx in rows:
        if "Perm" in label:
            print(f"{label:<28} {a:<10} {b:<10} {dx:<10}")

In [62]:
# Edita estos valores
rounds = 4
master_key_hex = '00112233445566778899'  # para 4 rondas se necesitan 5 subclaves
plain_a = 0x002B
plain_b = 0x0B2B

subkeys = spn16.expand_key_from_hex(master_key_hex, rounds)
cipher_a = spn16.spn_encrypt_block(plain_a, subkeys, rounds)
cipher_b = spn16.spn_encrypt_block(plain_b, subkeys, rounds)

print(f'K1..K{rounds+1} reales: {[fmt16(k) for k in subkeys]}')
print()
trace_a, trace_b = show_encryption_pair_trace(plain_a, plain_b, subkeys, rounds)
print()
print(f'Cifrado A = {fmt16(cipher_a)}')
print(f'Cifrado B = {fmt16(cipher_b)}')
print(f'Delta C  = {fmt16(cipher_a ^ cipher_b)}')


K1..K5 reales: ['0x0011', '0x2233', '0x4455', '0x6677', '0x8899']

Cifrado del par con SPN16
Fase                         Texto A    Texto B    XOR       
----------------------------------------------------------------
Texto claro                  0x002B     0x0B2B     0x0B00    
R1 despues XOR K1            0x003A     0x0B3A     0x0B00    
R1 despues S-box             0xEE16     0xEC16     0x0200    
R1 despues Perm              0xCDD2     0xCD92     0x0040    
R2 despues XOR K2            0xEFE1     0xEFA1     0x0040    
R2 despues S-box             0x0704     0x0764     0x0060    
R2 despues Perm              0x0544     0x0764     0x0220    
R3 despues XOR K3            0x4111     0x4331     0x0220    
R3 despues S-box             0x2444     0x2114     0x0550    
R3 despues Perm              0x0780     0x0186     0x0606    
R4 despues XOR K4            0x61F7     0x67F1     0x0606    
R4 despues S-box             0xB478     0xB874     0x0C0C    
Cifrado despues XOR K5       0x3CE1 

## Reversa de la ultima ronda

Cambia `candidate_k_r` y `candidate_k_final` por las subclaves candidatas que quieras probar.
Si quieres verificar con la clave correcta, deja los valores reales por defecto.


In [63]:
# Prueba aqui tus subclaves candidatas para la ultima ronda
candidate_k_r = subkeys[rounds - 1]   # cambia este valor si quieres probar otra clave
candidate_k_final = subkeys[rounds]   # cambia este valor si quieres probar otra clave

print(f'K{rounds} candidata     = {fmt16(candidate_k_r)}')
print(f'K{rounds+1} candidata   = {fmt16(candidate_k_final)}')
print()
reverse_a, reverse_b = show_reverse_last_round_pair(
    cipher_a,
    cipher_b,
    candidate_k_r,
    candidate_k_final,
    rounds,
)


K4 candidata     = 0x6677
K5 candidata   = 0x8899

Reversa de la ultima ronda con subclaves candidatas
Fase                         Texto A    Texto B    XOR       
----------------------------------------------------------------
Texto cifrado                0x3CE1     0x30ED     0x0C0C    
Deshacer XOR K5 candidato    0xB478     0xB874     0x0C0C    
Aplicar S-box^-1 de R4       0x61F7     0x67F1     0x0606    
Deshacer XOR K4 candidato    0x0780     0x0186     0x0606    


## Busqueda intensiva

In [64]:
# Buscar pares P, P' con Delta P = 0x0B00 y trayectoria:
# 0B00 -> 0040 -> 0220 -> 0606 en Texto claro, R1 Perm, R2 Perm, R3 Perm.

rounds = 4
master_key_hex = '00112233445566778899'
subkeys = spn16.expand_key_from_hex(master_key_hex, rounds)

target_deltas = [0x0B00, 0x0040, 0x0220, 0x0606]
max_pairs = 10


def perm_deltas_for_pair(p1: int, p2: int):
    trace_a = trace_encrypt_block(p1, subkeys, rounds)
    trace_b = trace_encrypt_block(p2, subkeys, rounds)
    deltas = []
    for (label_a, value_a), (_, value_b) in zip(trace_a, trace_b):
        if label_a == 'Texto claro' or 'despues Perm' in label_a:
            deltas.append(value_a ^ value_b)
    return deltas


found_pairs = []
for plain_a in range(1 << 16):
    plain_b = plain_a ^ target_deltas[0]
    if plain_a > plain_b:
        continue

    if perm_deltas_for_pair(plain_a, plain_b) == target_deltas:
        cipher_a = spn16.spn_encrypt_block(plain_a, subkeys, rounds)
        cipher_b = spn16.spn_encrypt_block(plain_b, subkeys, rounds)
        found_pairs.append((plain_a, plain_b, cipher_a, cipher_b))
        if len(found_pairs) >= max_pairs:
            break

print('Trayectoria objetivo:', ' -> '.join(f'{d:04X}' for d in target_deltas))
print(f'Pares encontrados: {len(found_pairs)}')

for i, (p1, p2, c1, c2) in enumerate(found_pairs, start=1):
    print(
        f'{i:02d}. P={fmt16(p1)}  P\'={fmt16(p2)}  '
        f'C={fmt16(c1)}  C\'={fmt16(c2)}  Delta C={fmt16(c1 ^ c2)}'
    )

if found_pairs:
    print('\nTraza del primer par encontrado:')
    plain_a, plain_b, cipher_a, cipher_b = found_pairs[0]
    trace_a, trace_b = show_encryption_pair_trace(plain_a, plain_b, subkeys, rounds, only_perm=True)


Trayectoria objetivo: 0B00 -> 0040 -> 0220 -> 0606
Pares encontrados: 10
01. P=0x002B  P'=0x0B2B  C=0x3CE1  C'=0x30ED  Delta C=0x0C0C
02. P=0x005E  P'=0x0B5E  C=0xF1EF  C'=0xF4EC  Delta C=0x0503
03. P=0x0064  P'=0x0B64  C=0x8629  C'=0x832A  Delta C=0x0503
04. P=0x006B  P'=0x0B6B  C=0x3AF4  C'=0x35FB  Delta C=0x0F0F
05. P=0x0071  P'=0x0B71  C=0x0115  C'=0x0410  Delta C=0x0505
06. P=0x0084  P'=0x0B84  C=0x8725  C'=0x8920  Delta C=0x0E05
07. P=0x008B  P'=0x0B8B  C=0x3CF1  C'=0x30FD  Delta C=0x0C0C
08. P=0x009B  P'=0x0B9B  C=0x3AE4  C'=0x35EB  Delta C=0x0F0F
09. P=0x00C1  P'=0x0BC1  C=0x0B2E  C'=0x0823  Delta C=0x030D
10. P=0x00FE  P'=0x0BFE  C=0xF29E  C'=0xFF93  Delta C=0x0D0D

Traza del primer par encontrado:
Cifrado del par con SPN16
Fase                         Texto A    Texto B    XOR       
----------------------------------------------------------------
Texto claro                  0x002B     0x0B2B     0x0B00    
R1 despues Perm              0xCDD2     0xCD92     0x0040    
R2 des